# RAW DATA INTERACTIVE WRANGLER
1. Data Wrangling Briefing
2. Data Wrangling Setup
3. Wrangling Data Tables of Instagram 
4. Wrangling Data Tables of TikTok
5. Wrangling Data Charts of Instagram and TikTok

## 1. Data Wrangling Briefing:
During the evaluation from viewing the dataframes and screenshots of the original data, it was possible to gather the following details.

Wrangling Tasks:
- Eliminate datapoints that contain only the character "|_|"
- Remove NaN datapoints
- Transform one column into several using "|_|" as a delimiter.
- Set appropriate column headers
- Transform all numbers into floats

Additionally, the following raw datasets are redundant or decontextualized, and should be disregarded for the next steps:
- raw_ig_table_1
- raw_ig_table_2
- raw_ig_table_3
- raw_ig_chart_2
- raw_tk_table_1
- raw_tk_table_2
- raw_tk_table_4
- raw_tk_table_5
- raw_tk_chart_3

## 2. Data Wrangling Setup:

In [ ]:
import os
import sys
import sqlite3

import pandas as pd

actl_dir = os.getcwd()
root_dir = os.path.dirname(actl_dir)
sys.path.append(root_dir)

from src.utils.ipynb import(
    DataFrameVisualizer,
    get_raw_db_path)

view = DataFrameVisualizer.viewer

raw_db = get_raw_db_path(root_dir)
conn = sqlite3.connect(raw_db)

In [ ]:
tables = conn.execute("SELECT name FROM sqlite_master WHERE type='table';").fetchall()
tables = [table[0] for table in tables]

dfs = {'ig_table': [], 'ig_chart': [], 'tk_table': [], 'tk_chart': []}

for table in tables:
    for key in dfs.keys():
        if f'raw_{key}' in table:
            dfs[key].append(pd.read_sql_query(f'SELECT * FROM {table}', conn))

ig_table_df = pd.concat(dfs['ig_table'], ignore_index=True)
ig_chart_df = pd.concat(dfs['ig_chart'], ignore_index=True)
tk_table_df = pd.concat(dfs['tk_table'], ignore_index=True)
tk_chart_df = pd.concat(dfs['tk_chart'], ignore_index=True)

raw_ig_table_0 = ig_table_df.iloc[:, [1]].copy()
raw_ig_table_4 = ig_table_df.iloc[:, [5]].copy()
raw_ig_table_5 = ig_table_df.iloc[:, [6]].copy()
raw_ig_table_6 = ig_table_df.iloc[:, [7]].copy()
raw_ig_chart_0 = ig_chart_df.iloc[:, [1]].copy()
raw_ig_chart_1 = ig_chart_df.iloc[:, [2]].copy()

raw_tk_table_0 = tk_table_df.iloc[:, [1]].copy()
raw_tk_table_3 = tk_table_df.iloc[:, [4]].copy()
raw_tk_table_6 = tk_table_df.iloc[:, [7]].copy()
raw_tk_table_7 = tk_table_df.iloc[:, [8]].copy()
raw_tk_chart_0 = tk_chart_df.iloc[:, [1]].copy()
raw_tk_chart_1 = tk_chart_df.iloc[:, [2]].copy()
raw_tk_chart_2 = tk_chart_df.iloc[:, [3]].copy()

ig_tables = [raw_ig_table_0, raw_ig_table_4, raw_ig_table_5, raw_ig_table_6]
ig_charts = [raw_ig_chart_0, raw_ig_chart_1]

tk_tables = [raw_tk_table_0, raw_tk_table_6, raw_tk_table_7]
tk_charts = [raw_tk_chart_0, raw_tk_chart_1, raw_tk_chart_2]

## 3. Wrangling Data Tables of Instagram:

In [ ]:
def split_values(val):
    return pd.Series(val.split('|_|', 1))

for table in ig_tables:
    table.dropna(inplace=True)
    table[['category', 'value']] = table[table.columns[0]].apply(split_values)
    table.drop(table.columns[0], axis=1, inplace=True)
    table.drop(table.index[0], inplace=True)
raw_ig_table_4.rename(columns={'category': 'region'}, inplace=True)
raw_ig_table_5.rename(columns={'category': 'age_bk'}, inplace=True)
raw_ig_table_6.rename(columns={'category': 'gender'}, inplace=True) 

## 4. Wrangling Data Tables of TikTok:

In [ ]:
def split_values(val):
    return pd.Series(val.split('|_|', 1))

for table in tk_tables:
    table.dropna(inplace=True)
    table[['category', 'value']] = table[table.columns[0]].apply(split_values)
    if table['value'].str.contains('%').any():
        table['value'] = table['value'].str.split('%').str[0]
    table.drop(table.columns[0], axis=1, inplace=True)
    table.drop(table.index[0], inplace=True)
raw_tk_table_6.rename(columns={'category': 'age_bk'}, inplace=True)
raw_tk_table_7.rename(columns={'category': 'gender'}, inplace=True) 

In [ ]:
for index, (ig_table, tk_table) in enumerate(zip(ig_tables, tk_tables)):
    if index > 0:
        ig_table['value'] = ig_table['value'].astype('float64')
        tk_table['value'] = tk_table['value'].astype('float64')

In [ ]:
raw_tk_table_3['asia_pacific'] = None
raw_tk_table_3['north_america'] = None
raw_tk_table_3['europe'] = None
raw_tk_table_3['latin_america'] = None
raw_tk_table_3['middle_east_northern_africa'] = None

In [ ]:
raw_tk_table_3